# **NLP Practical 6 — Text Classification & Sentiment Analysis**

---

**Name: Shubham Sarwar**  
**PRN: 202301040127**

**Aim:** Build the **Bag of Words (BoW)** representation of a text, visualise word
frequencies (bar chart, heatmap, word cloud), and then apply **TF-IDF + Naïve Bayes**
to perform sentiment classification of movie reviews as positive or negative.

### Key Concepts (from the practical sheet)
- **Bag of Words (BoW):** represents text using word frequencies, ignoring grammar and order.
- **TF-IDF:** assigns importance to words based on frequency in one document and rarity across the corpus.
- **Naïve Bayes:** probabilistic classifier based on Bayes' theorem, assuming feature independence.


## **Part A — Bag of Words Model**

## **1. Install / Import Libraries**

In [ ]:
!pip install nltk wordcloud seaborn scikit-learn

In [ ]:
import nltk
import re
import heapq
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from wordcloud import WordCloud

# scikit-learn pieces for TF-IDF and Naive Bayes (used in Part B)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Download data files NLTK needs
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

## **2. Text Corpus**

In [ ]:
text = """Beans. I was trying to explain to somebody as we were flying in, that's corn.
That's beans. And they were very impressed at my agricultural knowledge.
Please give it up for Amaury once again for that outstanding introduction.
I have a bunch of good friends here today, including somebody who I served with,
who is one of the finest senators in the country, and we're lucky to have him,
your Senator, Dick Durbin is here. I also noticed, by the way,
former Governor Edgar here, who I haven't seen in a long time, and
somehow he has not aged and I have. And it's great to see you, Governor.
I want to thank President Killeen and everybody at the U of I System for
making it possible for me to be here today. And I am deeply honored at the Paul
Douglas Award that is being given to me. He is somebody who set the path for so
much outstanding public service here in Illinois. Now, I want to start by
addressing the elephant in the room. I know people are still wondering why
I didn't speak at the commencement."""

## **3. Step 1 — Preprocessing the Text**

Before applying the BoW model we clean the text:
1. Convert to lowercase.
2. Remove non-word characters (numbers, punctuation, symbols).
3. Collapse multiple spaces into one.

We split the text into individual sentences first because each sentence will become
one row in the Bag-of-Words matrix later.


In [ ]:
# Split the whole text into individual sentences using NLTK's sentence tokenizer
dataset = nltk.sent_tokenize(text)

# Clean every sentence: lowercase, drop non-word chars, squeeze spaces
for i in range(len(dataset)):
    dataset[i] = dataset[i].lower()                       # lowercase
    dataset[i] = re.sub(r'\W', ' ', dataset[i])           # \W = anything not a-z 0-9 _
    dataset[i] = re.sub(r'\s+', ' ', dataset[i])          # collapse multiple spaces

# Show the cleaned sentences
print("Cleaned sentences:\n")
for i, sentence in enumerate(dataset):
    print(f"Sentence {i+1}: {sentence}")

## **4. Step 2 — Counting Word Frequencies**

We loop through each sentence, tokenize into words, and count how many times each
word appears in the whole corpus. Result is stored in a dictionary `word2count`.


In [ ]:
word2count = {}

for data in dataset:
    words = nltk.word_tokenize(data)
    for word in words:
        # if first time we see this word, set count = 1; else increment
        if word not in word2count:
            word2count[word] = 1
        else:
            word2count[word] += 1

# Remove English stop-words so the table focuses on meaningful words
stop_words = set(stopwords.words('english'))
filtered_word2count = {w: c for w, c in word2count.items() if w not in stop_words}

# Put into a DataFrame for a clean tabular display, sorted by frequency
word_freq_df = pd.DataFrame(
    list(filtered_word2count.items()),
    columns=['Word', 'Frequency']
).sort_values(by='Frequency', ascending=False).reset_index(drop=True)

print("Word Frequency Table (top 20 after stop-word removal):\n")
print(word_freq_df.head(20).to_string(index=False))

## **5. Step 3 — Selecting the Most Frequent Words (Bar Chart)**

We pick the top 10 most frequent (non-stop) words and visualise them. These will
form the columns of our BoW matrix in the next step.


In [ ]:
# heapq.nlargest is a fast way to get the top N items from a dictionary by value
freq_words = heapq.nlargest(10, filtered_word2count, key=filtered_word2count.get)
print(f"Top 10 frequent words: {freq_words}\n")

# Bar chart
top_words = sorted(filtered_word2count.items(), key=lambda x: x[1], reverse=True)[:10]
words, counts = zip(*top_words)

plt.figure(figsize=(10, 6))
plt.bar(words, counts, color='skyblue', edgecolor='steelblue')
plt.xticks(rotation=45, ha='right')
plt.title('Top 10 Most Frequent Words')
plt.xlabel('Words')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

## **6. Step 4 — Building the Bag-of-Words Matrix (Heatmap)**

The BoW matrix is a **binary matrix**:
- Each **row** is a sentence.
- Each **column** is one of the top-10 frequent words.
- A cell is **1** if the word appears in that sentence, otherwise **0**.

We visualise this matrix as a heat-map — green = word present, red = absent.


In [ ]:
X = []

for data in dataset:
    vector = []
    words_in_sent = nltk.word_tokenize(data)
    for word in freq_words:
        if word in words_in_sent:
            vector.append(1)
        else:
            vector.append(0)
    X.append(vector)

X = np.asarray(X)
print(f"BoW matrix shape: {X.shape}  ({X.shape[0]} sentences × {X.shape[1]} frequent words)\n")

# Heatmap visualisation
plt.figure(figsize=(12, 7))
sns.heatmap(
    X,
    cmap='RdYlGn',                # red = 0, green = 1
    cbar=False,
    annot=True, fmt="d",
    linewidths=0.5, linecolor='grey',
    xticklabels=freq_words,
    yticklabels=[f"Sentence {i+1}" for i in range(len(dataset))]
)
plt.title('Bag of Words Matrix (1 = word present, 0 = absent)')
plt.xlabel('Frequent Words')
plt.ylabel('Sentences')
plt.tight_layout()
plt.show()

## **7. Step 5 — Word Cloud Visualization**

In a word cloud, each word's **size is proportional to its frequency** — the biggest
words are the most common. This is a quick visual summary of the entire text.


In [ ]:
wordcloud = WordCloud(
    width=800,
    height=400,
    background_color='white',
    stopwords=stop_words
).generate_from_frequencies(filtered_word2count)

plt.figure(figsize=(10, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.title("Word Cloud of Frequent Words")
plt.show()

---

## **Part B — Sentiment Classification using TF-IDF + Naïve Bayes**

The Bag-of-Words model above treats every word as a binary feature (present/absent).
**TF-IDF** improves on this by giving more weight to words that are frequent in a
document but rare across the whole corpus — these are the most informative words.

We will use TF-IDF features together with a **Multinomial Naïve Bayes** classifier
to predict whether a movie review is positive or negative.


## **8. Build a Small Movie-Review Dataset**

In [ ]:
# Each tuple is (review_text, label). In a real project you'd load IMDb / Rotten Tomatoes.
data = [
    # ---------- Positive reviews ----------
    ("The movie was fantastic and brilliant, I loved every minute.", "positive"),
    ("Excellent acting and a wonderful story, must watch!", "positive"),
    ("Great direction, amazing performance and beautiful music.", "positive"),
    ("An outstanding film with a powerful and emotional story.", "positive"),
    ("Loved the cinematography and the actors did a superb job.", "positive"),
    ("A heartwarming and inspiring movie, truly enjoyable.", "positive"),
    ("Best film I have seen this year, absolutely brilliant.", "positive"),
    ("The screenplay is wonderful and the direction is top notch.", "positive"),
    ("Funny, touching and well-acted, highly recommend it.", "positive"),
    ("A masterpiece of cinema, every scene was memorable.", "positive"),
    ("Incredible visuals and an engaging plot from start to finish.", "positive"),
    ("Beautiful soundtrack and the cast delivered amazing performances.", "positive"),
    ("This film made me laugh and cry, just superb storytelling.", "positive"),
    ("A truly enjoyable experience with great direction.", "positive"),
    ("Loved every moment, the lead actor was outstanding.", "positive"),
    ("A delightful and refreshing movie, would watch again.", "positive"),
    ("The story is gripping and the action scenes are thrilling.", "positive"),
    ("Such a wonderful, well-crafted film, highly recommended.", "positive"),
    ("Brilliant performance and clever script make it a must watch.", "positive"),
    ("Heartfelt and beautifully shot, a near-perfect film.", "positive"),

    # ---------- Negative reviews ----------
    ("Terrible movie, boring and a complete waste of time.", "negative"),
    ("The plot was dull and predictable, very disappointing.", "negative"),
    ("Bad acting, weak story and poor direction throughout.", "negative"),
    ("Awful film, I wanted to leave after the first ten minutes.", "negative"),
    ("Worst movie of the year, totally not worth watching.", "negative"),
    ("Horrible script and the actors were terrible.", "negative"),
    ("Boring, long and pointless, do not recommend.", "negative"),
    ("Disappointing sequel, nothing was good about it.", "negative"),
    ("Poorly made movie, bad effects and bad acting.", "negative"),
    ("Slow paced and uninteresting, an absolute disaster.", "negative"),
    ("Confusing storyline and weak characters, very bad film.", "negative"),
    ("The dialogues were cringe and the acting was wooden.", "negative"),
    ("Nothing happens for most of the movie, complete bore.", "negative"),
    ("Terrible direction, the worst film I have ever seen.", "negative"),
    ("Predictable, cliched and boring, a huge letdown.", "negative"),
    ("Bad screenplay, lazy direction and zero emotions.", "negative"),
    ("Annoying characters and a stupid ending, awful movie.", "negative"),
    ("This film was a painful experience, dull and slow.", "negative"),
    ("Cheap effects and a ridiculous plot ruin the movie.", "negative"),
    ("Disappointing, no good acting and a very weak script.", "negative"),
]

df = pd.DataFrame(data, columns=["review", "sentiment"])
print(f"Total reviews: {len(df)}")
print(df["sentiment"].value_counts())

## **9. Clean the Reviews**

In [ ]:
def clean_text(t):
    t = t.lower()
    t = re.sub(r"[^a-z\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["clean"] = df["review"].apply(clean_text)
df.head()

## **10. Split into Train / Test Sets**

We use 75% for training, 25% for testing, with `stratify=y` to keep the same
positive : negative ratio in both splits.


In [ ]:
X = df["clean"].values
y = df["sentiment"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Training samples: {len(X_train)}")
print(f"Testing samples : {len(X_test)}")

## **11. Convert Text to TF-IDF Vectors**

$$\mathrm{TF}(t,d)=\frac{\text{count of } t \text{ in } d}{\text{words in } d}, \qquad
\mathrm{IDF}(t)=\log\frac{N}{1+df(t)}$$

$$\mathrm{TF\text{-}IDF}(t,d) = \mathrm{TF}(t,d) \times \mathrm{IDF}(t)$$

A word that is **frequent in one review** (high TF) but **rare across all reviews**
(high IDF) carries the most discriminative information.


In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english",       # drop common stop-words
    ngram_range=(1, 2),         # use single words AND word pairs ("not good")
    min_df=1
)

# fit_transform LEARNS vocabulary AND vectorises the training set
X_train_tfidf = vectorizer.fit_transform(X_train)

# transform only USES the already-learned vocabulary for the test set
# (must NOT fit again — that would leak test info)
X_test_tfidf  = vectorizer.transform(X_test)

print(f"TF-IDF matrix shape (train): {X_train_tfidf.shape}")
print(f"Vocabulary size            : {len(vectorizer.vocabulary_)}")

## **12. Train Multinomial Naïve Bayes**

Bayes' theorem gives us:

$$P(\text{class} \mid \text{words}) \propto P(\text{class}) \prod_i P(\text{word}_i \mid \text{class})$$

The classifier multiplies these probabilities for each class and picks the higher one.


In [ ]:
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# Predict on the held-out test set
y_pred = model.predict(X_test_tfidf)

print("Sample test predictions:")
for review, true, pred in list(zip(X_test, y_test, y_pred))[:5]:
    mark = "✓" if true == pred else "✗"
    print(f"  [{mark}] true={true:8s} pred={pred:8s}  '{review[:60]}...'")

## **13. Evaluate: Accuracy + Confusion Matrix**

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy = {acc*100:.2f}%\n")

print("Classification report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=["positive", "negative"])
print("Confusion Matrix:")
print(cm)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["positive", "negative"],
            yticklabels=["positive", "negative"], cbar=False)
plt.title("Sentiment Classifier — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## **14. Predict on Brand-New Reviews**

In [ ]:
new_reviews = [
    "I really loved this brilliant film, the acting was amazing!",
    "Horrible boring disaster, total waste of time.",
    "An emotional masterpiece with stunning visuals.",
    "Boring, slow, dull, with bad direction and weak script.",
]
cleaned = [clean_text(r) for r in new_reviews]
preds   = model.predict(vectorizer.transform(cleaned))
probs   = model.predict_proba(vectorizer.transform(cleaned))

print("Predictions on new reviews:\n")
for review, label, prob in zip(new_reviews, preds, probs):
    conf = max(prob)
    print(f"  [{label.upper():8s}] (confidence {conf:.2f})  '{review}'")

---

### **Conclusion**

In this practical we:

- Built a **Bag-of-Words** representation, visualised it with a bar chart, heatmap
  and word cloud (Part A).
- Used **TF-IDF** to assign importance to words and trained a **Multinomial Naïve
  Bayes** classifier to predict sentiment with high accuracy (Part B).

**Real-world uses:** Amazon / IMDb / Flipkart review analytics, spam detection,
toxic-comment filtering, news topic classification, customer-support ticket routing.
